<a href="https://colab.research.google.com/github/T1nTin48/SDS320-project/blob/main/V4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q segmentation-models-pytorch rasterio geopandas tqdm albumentations

import os
import zipfile
import glob
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.merge import merge
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from tqdm.notebook import tqdm
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Aktive Hardware: {device}")

# ZIP entpacken (angepasst auf result_V4.zip)
zip_path = '/content/result_V4.zip'
extract_dir = '/content/data_V4'
os.makedirs(extract_dir, exist_ok=True)

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    # Prüfen, ob ein Unterordner (z.B. result_V4/) im ZIP enthalten war
    items = [i for i in os.listdir(extract_dir) if not i.startswith('__MACOSX') and not i.startswith('.')]
    if len(items) == 1 and os.path.isdir(os.path.join(extract_dir, items[0])):
        data_dir = os.path.join(extract_dir, items[0])
    else:
        data_dir = extract_dir

    print(f"✅ Datenordner erfolgreich gesetzt auf: {data_dir}")
    gefundene_dateien = sorted([f for f in os.listdir(data_dir) if not f.startswith('.')])
    print("Gefundene Dateien:", gefundene_dateien)
else:
    print("⚠️ Datei '/content/result_V4.zip' nicht gefunden. Bitte links im Datei-Tab hochladen.")

Aktive Hardware: cuda
✅ Datenordner erfolgreich gesetzt auf: /content/data_V4
Gefundene Dateien: ['1m_Kreis11_2013.tif', '1m_Kreis11_2013.tif.aux.xml', '1m_Kreis11_2024.tif', '1m_Kreis11_2024.tif.aux.xml', '1m_Kreis9_2013.tif', '1m_Kreis9_2013.tif.aux.xml', '1m_Kreis9_2024.tif', '1m_Kreis9_2024.tif.aux.xml', 'Split_Blocs_Kreis11.gpkg', 'Split_Blocs_Kreis9_400m.gpkg', 'rasterized_Kreis11_2013_1m.tif', 'rasterized_Kreis11_2013_1m.tif.aux.xml', 'rasterized_Kreis11_2024_1m_richtig.tif', 'rasterized_Kreis11_2024_1m_richtig.tif.aux.xml', 'rasterized_Kreis9_2013_1m.tif', 'rasterized_Kreis9_2013_1m.tif.aux.xml', 'rasterized_Kreis9_2024_1m_richtig.tif', 'rasterized_Kreis9_2024_1m_richtig.tif.aux.xml']


In [ ]:
from rasterio.windows import Window

def create_sliding_chips(ortho_path, mask_path, output_base_dir, dataset_name, chip_size=400, stride=200):
    out_img_dir = os.path.join(output_base_dir, dataset_name, 'images')
    out_mask_dir = os.path.join(output_base_dir, dataset_name, 'masks')
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)

    valid_tiles_count = 0

    with rasterio.open(ortho_path) as src_img, rasterio.open(mask_path) as src_mask:
        assert src_img.shape == src_mask.shape, f"Shape-Mismatch bei {dataset_name}"
        H, W = src_img.height, src_img.width

        meta_img = src_img.meta.copy()
        meta_mask = src_mask.meta.copy()
        meta_img.update({"driver": "GTiff", "height": chip_size, "width": chip_size})
        meta_mask.update({"driver": "GTiff", "height": chip_size, "width": chip_size})

        idx = 0
        for row in range(0, H, stride):
            for col in range(0, W, stride):
                win_h = min(chip_size, H - row)
                win_w = min(chip_size, W - col)
                window = Window(col, row, win_w, win_h)

                out_img = src_img.read(window=window)
                out_lbl = src_mask.read(1, window=window)

                fixed_img = np.zeros((src_img.count, chip_size, chip_size), dtype=out_img.dtype)
                fixed_lbl = np.full((1, chip_size, chip_size), 255, dtype=out_lbl.dtype)

                fixed_img[:, :win_h, :win_w] = out_img
                fixed_lbl[0, :win_h, :win_w] = out_lbl

                # Kacheln ohne gültige Pixel (nur NoData 255) überspringen
                if np.all(fixed_lbl == 255):
                    continue

                win_transform = src_img.window_transform(window)
                meta_img.update({"transform": win_transform})
                meta_mask.update({"transform": win_transform})

                img_filename = os.path.join(out_img_dir, f"{dataset_name}_tile_{idx:04d}.tif")
                mask_filename = os.path.join(out_mask_dir, f"{dataset_name}_tile_{idx:04d}.tif")

                with rasterio.open(img_filename, "w", **meta_img) as dest_img:
                    dest_img.write(fixed_img)
                with rasterio.open(mask_filename, "w", **meta_mask) as dest_mask:
                    dest_mask.write(fixed_lbl)

                valid_tiles_count += 1
                idx += 1

    print(f"✅ {dataset_name}: {valid_tiles_count} überlappende Kacheln (Stride {stride}px) gespeichert")

dataset_dir_v5 = '/content/dataset_V5'

# Kreis 9 (Training): Stride 200 -> 4x mehr Trainingsbeispiele & diverse Ausschnitte
create_sliding_chips(os.path.join(data_dir, '1m_Kreis9_2013.tif'), os.path.join(data_dir, 'rasterized_Kreis9_2013_1m.tif'), dataset_dir_v5, 'Kreis9_2013', stride=200)
create_sliding_chips(os.path.join(data_dir, '1m_Kreis9_2024.tif'), os.path.join(data_dir, 'rasterized_Kreis9_2024_1m_richtig.tif'), dataset_dir_v5, 'Kreis9_2024', stride=200)

# Kreis 11 (Test & Mosaik): Stride 350 (50px Überlapp) -> eliminiert die weissen Gitterlinien komplett
create_sliding_chips(os.path.join(data_dir, '1m_Kreis11_2013.tif'), os.path.join(data_dir, 'rasterized_Kreis11_2013_1m.tif'), dataset_dir_v5, 'Kreis11_2013', stride=350)
create_sliding_chips(os.path.join(data_dir, '1m_Kreis11_2024.tif'), os.path.join(data_dir, 'rasterized_Kreis11_2024_1m_richtig.tif'), dataset_dir_v5, 'Kreis11_2024', stride=350)

✅ Kreis9_2013: 401 überlappende Kacheln (Stride 200px) gespeichert
✅ Kreis9_2024: 401 überlappende Kacheln (Stride 200px) gespeichert
✅ Kreis11_2013: 144 überlappende Kacheln (Stride 350px) gespeichert
✅ Kreis11_2024: 144 überlappende Kacheln (Stride 350px) gespeichert


In [ ]:
# Starke Farb-/Helligkeits-Augmentation, damit braune/helle Böden nicht als versiegelt gelten
train_transform_v5 = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.HueSaturationValue(hue_shift_limit=25, sat_shift_limit=35, val_shift_limit=30, p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_transform_v5 = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

class SealedSurfaceDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.img_paths[idx]) as src:
            img = np.transpose(src.read()[:3], (1, 2, 0)).astype(np.uint8)
        with rasterio.open(self.mask_paths[idx]) as src:
            mask_raw = src.read(1).astype(np.int64)

        if self.transform:
            augmented = self.transform(image=img, mask=mask_raw)
            img = augmented['image']
            mask_raw = augmented['mask']

        img_tensor = torch.from_numpy(np.transpose(img, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask_raw).long()
        return img_tensor, mask_tensor

# Alle Kreis-9-Kacheln sammeln und 90/10 in Train / Val (nur Kreis 9!) splitten
all_imgs, all_masks = [], []
for folder in [os.path.join(dataset_dir_v5, 'Kreis9_2013'), os.path.join(dataset_dir_v5, 'Kreis9_2024')]:
    for img_p in sorted(glob.glob(os.path.join(folder, 'images', '*.tif'))):
        mask_p = os.path.join(folder, 'masks', os.path.basename(img_p))
        if os.path.exists(mask_p):
            all_imgs.append(img_p)
            all_masks.append(mask_p)

rng = np.random.default_rng(42)
indices = rng.permutation(len(all_imgs))
split_idx = int(0.9 * len(indices))

train_idx, val_idx = indices[:split_idx], indices[split_idx:]
train_dataset = SealedSurfaceDataset([all_imgs[i] for i in train_idx], [all_masks[i] for i in train_idx], transform=train_transform_v5)
val_dataset = SealedSurfaceDataset([all_imgs[i] for i in val_idx], [all_masks[i] for i in val_idx], transform=val_transform_v5)

train_loader = DataLoader(train_dataset, batch_size=12, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=12, shuffle=False)

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=2
).to(device)

ce_loss = nn.CrossEntropyLoss(ignore_index=255)
dice_loss = smp.losses.DiceLoss(mode='multiclass', ignore_index=255)

def combined_loss(pred, target):
    return 0.5 * ce_loss(pred, target) + 0.5 * dice_loss(pred, target)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

print(f"Starte V5-Training auf {len(train_dataset)} Kreis-9-Kacheln ({len(val_dataset)} Val-Kacheln)...")

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoche {epoch+1:02d}/{epochs:02d} | Loss: {epoch_loss/len(train_loader):.4f}")

# Optimalen Schwellenwert auf Kreis-9-Validierung bestimmen
model.eval()
val_probs, val_targets = [], []
with torch.no_grad():
    for imgs, masks in val_loader:
        probs = torch.softmax(model(imgs.to(device)), dim=1)[:, 1, :, :].cpu()
        valid = (masks != 255)
        val_probs.append(probs[valid])
        val_targets.append(masks[valid])

val_probs = torch.cat(val_probs)
val_targets = torch.cat(val_targets)

best_thresh, best_f1 = 0.5, 0.0
for th in np.arange(0.30, 0.65, 0.05):
    pred_bin = (val_probs >= th).long()
    tp = ((pred_bin == 1) & (val_targets == 1)).sum().item()
    fp = ((pred_bin == 1) & (val_targets == 0)).sum().item()
    fn = ((pred_bin == 0) & (val_targets == 1)).sum().item()
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-6)
    if f1 > best_f1:
        best_f1, best_thresh = f1, float(th)

print(f"✅ V5-Training abgeschlossen! Optimaler Schwellenwert (aus Kreis-9-Val): {best_thresh:.2f} (Val-F1: {best_f1:.4f})")
torch.save(model.state_dict(), '/content/unet_resnet34_kreis9_v5.pth')

Starte V5-Training auf 721 Kreis-9-Kacheln (81 Val-Kacheln)...
Epoche 01/20 | Loss: 0.3556
Epoche 05/20 | Loss: 0.2577
Epoche 10/20 | Loss: 0.2232
Epoche 15/20 | Loss: 0.1994
Epoche 20/20 | Loss: 0.1934
✅ V5-Training abgeschlossen! Optimaler Schwellenwert (aus Kreis-9-Val): 0.45 (Val-F1: 0.9341)


In [ ]:
dir_2013 = os.path.join(dataset_dir_v5, 'Kreis11_2013', 'images')
dir_2024 = os.path.join(dataset_dir_v5, 'Kreis11_2024', 'images')
mask_dir_2013 = os.path.join(dataset_dir_v5, 'Kreis11_2013', 'masks')
mask_dir_2024 = os.path.join(dataset_dir_v5, 'Kreis11_2024', 'masks')

model.eval()
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def predict_full_raster(ortho_path, mask_path, chip_size=400, stride=350):
    with rasterio.open(ortho_path) as src_img, rasterio.open(mask_path) as src_mask:
        H, W = src_img.height, src_img.width
        prob_sum = np.zeros((H, W), dtype=np.float32)
        weight_sum = np.zeros((H, W), dtype=np.float32)
        ref_mask = src_mask.read(1)
        meta = src_mask.meta.copy()

        # Weiches Gewichtungsfenster für nahtlose Übergänge ohne Gitterlinien
        wy = np.hanning(chip_size)[:, None] + 0.05
        wx = np.hanning(chip_size)[None, :] + 0.05
        weight_patch = (wy * wx).astype(np.float32)

        with torch.no_grad():
            for row in range(0, H, stride):
                for col in range(0, W, stride):
                    win_h = min(chip_size, H - row)
                    win_w = min(chip_size, W - col)
                    window = Window(col, row, win_w, win_h)

                    out_img = src_img.read(window=window)[:3].astype(np.float32) / 255.0
                    fixed_img = np.zeros((3, chip_size, chip_size), dtype=np.float32)
                    fixed_img[:, :win_h, :win_w] = out_img

                    t_img = (torch.from_numpy(fixed_img).unsqueeze(0).to(device) - mean) / std
                    prob = torch.softmax(model(t_img), dim=1)[0, 1].cpu().numpy()

                    w_crop = weight_patch[:win_h, :win_w]
                    prob_sum[row:row+win_h, col:col+win_w] += prob[:win_h, :win_w] * w_crop
                    weight_sum[row:row+win_h, col:col+win_w] += w_crop

    prob_full = prob_sum / np.maximum(weight_sum, 1e-6)
    valid = (ref_mask != 255)
    prob_full = np.where(valid, prob_full, -9999.0).astype(np.float32)
    return prob_full, ref_mask, meta

prob_2013, ref_2013_mask, meta_13 = predict_full_raster(os.path.join(data_dir, '1m_Kreis11_2013.tif'), os.path.join(data_dir, 'rasterized_Kreis11_2013_1m.tif'))
prob_2024, ref_2024_mask, meta_24 = predict_full_raster(os.path.join(data_dir, '1m_Kreis11_2024.tif'), os.path.join(data_dir, 'rasterized_Kreis11_2024_1m_richtig.tif'))

def eval_full(prob_map, ref_map, year, thresh):
    valid = (ref_map != 255)
    p = (prob_map[valid] >= thresh)
    m = (ref_map[valid] == 1)

    tp = np.logical_and(p, m).sum()
    fp = np.logical_and(p, ~m).sum()
    fn = np.logical_and(~p, m).sum()

    prec = tp / (tp + fp + 1e-6)
    rec = tp / (tp + fn + 1e-6)
    f1 = 2 * prec * rec / (prec + rec + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)

    print(f"--- V5 Evaluierung Kreis 11 ({year}, Threshold={thresh:.2f}) ---")
    print(f"Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | IoU: {iou:.4f}")
    print(f"Predicted Area (Klasse 1): {p.sum():,.0f} m² | Reference Area: {m.sum():,.0f} m²\n")
    return int(p.sum()), int(m.sum())

pred_13_area, ref_13_area = eval_full(prob_2013, ref_2013_mask, "2013", best_thresh)
pred_24_area, ref_24_area = eval_full(prob_2024, ref_2024_mask, "2024", best_thresh)

print("=== V5 Change Detection Kreis 11 (2013 -> 2024) ===")
print(f"Modell-Differenz:   {pred_24_area - pred_13_area:+,.0f} m² ({((pred_24_area - pred_13_area) / pred_13_area) * 100:+.2f} %)")
print(f"Referenz-Differenz: {ref_24_area - ref_13_area:+,.0f} m² ({((ref_24_area - ref_13_area) / ref_13_area) * 100:+.2f} %)")

--- V5 Evaluierung Kreis 11 (2013, Threshold=0.45) ---
Precision: 0.9125 | Recall: 0.8679 | F1: 0.8897 | IoU: 0.8013
Predicted Area (Klasse 1): 7,960,569 m² | Reference Area: 8,369,796 m²

--- V5 Evaluierung Kreis 11 (2024, Threshold=0.45) ---
Precision: 0.9053 | Recall: 0.8841 | F1: 0.8946 | IoU: 0.8092
Predicted Area (Klasse 1): 8,049,067 m² | Reference Area: 8,242,393 m²

=== V5 Change Detection Kreis 11 (2013 -> 2024) ===
Modell-Differenz:   +88,498 m² (+1.11 %)
Referenz-Differenz: -127,403 m² (-1.52 %)


In [ ]:
for out_name, prob_arr, meta in [
    ('Oerlikon_Prob_2013_V5.tif', prob_2013, meta_13),
    ('Oerlikon_Prob_2024_V5.tif', prob_2024, meta_24)
]:
    out_meta = meta.copy()
    out_meta.update({'count': 1, 'dtype': 'float32', 'nodata': -9999.0})
    out_path = os.path.join('/content', out_name)
    with rasterio.open(out_path, 'w', **out_meta) as dst:
        dst.write(prob_arr, 1)
    files.download(out_path)

print("✅ Nahtlose V5-Mosaike (ohne Kachelgrenzen!) gespeichert und Download gestartet.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Nahtlose V5-Mosaike (ohne Kachelgrenzen!) gespeichert und Download gestartet.
